# Building a Unigram Language Model in NLP - Japanese

## A Natural Language Processing Project

## by Sean Fletcher

### Overview

Used a unigram model to calculate the probability of each word within a provided japanese text corpus. This required me to reprocess the text by removing special characters and converting everything to lowercase to ensure uniformity. I split the text into individual words (unigrams) to analyze their frequencies. I then determined the probability of each word by dividing its frequency by the total number of words in the text, displaying the top 10 most probable words along with their corresponding probabilities.


### Importing Data

The japanese text corpus I worked with was Phillips Oppenheim's novel "入れかわった男". Since this book is in the public domain, it is available for use at no cost through the [Gutemberg Project](https://www.gutenberg.org/).

The goal of the project was to showcase the adaptability and strength of anagrams within different linguistic frameworks, encompassing any alphabets including Cyrillic, Roman (Latin), Greek, and of course Japanese scripts.


In [1]:
# Imported data
def GetReviewsList(path):
    with open(path, encoding='utf-8-sig') as f:
        lines = f.readlines()

    return lines

# Created a list named `oppenheim`, where each element is an individual line from the book
oppenheim = GetReviewsList('./pg34158.txt')

### Data Cleaning

Since the Japanese language does not utilize uppercase letters, our data cleaning efforts will primarily involve the removal of stop words and punctuation tokens. In traditional Japanese writing, spaces are not used to separate words, which is common in many Western languages like English or Portuguese. Japanese text is typically composed of continuous characters with no spaces between them. For this reason a space was added between every Japanese idiogram.

The text obtained from Project Gutenberg also contained sections in English. These sections were eliminated by employing the `is_japanese` function to ensure the dataset is consistent with the target language.

Created a function named `cleanText` that takes a list of text as input and returns a list of cleaned tokens. Each token in this list was stripped of punctuation, and free of ['stop words'](https://medium.com/@saitejaponugoti/stop-words-in-nlp-5b248dadad47).

The following are th lists of 'stop words' and punctuation used: 

'stop words': ['の', 'に', 'は', 'を', 'た', 'が', 'で', 'て', 'と', 'し', 'れ', 'さ', 'ある', 'いる', 'も', 'する', 'から', 'な', 'こと', 'として', 'い', 'や', 'する', 'など', 'なり', 'なく', 'まで', 'だ', 'へ', 'か', 'だっ', 'その', 'あっ', 'よう', 'また', 'もの', 'という', 'あり', 'まし', 'ませ', 'う', 'ない', 'ながら', 'なけれ', 'なし', 'ず', 'なっ', 'れる', 'られ', 'なる', 'べき', 'ほど', 'ます', 'てる', 'なら', 'せる', 'され', 'して']

punctuation: ['。', '、', '？', '！', '「', '」', '『', '』', '（', '）', '；', '：', '-']

Generally, Python's NLTK (Natural Language Toolkit) would be use for natural language processing, but it's not the best choice for tokenizing text in Japanese. Instead the library called `janome` was used for the Japanese text tokenization (`from janome.tokenizer import Tokenizer`).

In [3]:
import re
def is_japanese(word):

    # Regex to identify if the word contains any Japanese character
    # Hiragana: U+3040-U+309F, Katakana: U+30A0-U+30FF, Kanji: U+4E00-U+9FAF
    jap_regex = r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FAF]'
    
    # Regex to identify if the word contains any Latin alphabet character
    eng_regex = r'[a-zA-Z]'
    
    # If the word contains any Japanese characters and no Latin characters, it's considered Japanese
    if re.search(jap_regex, word) and not re.search(eng_regex, word):
        return True
    else:
        return False

In [5]:
# If janome is not already running on system, intall here:
# %pip install janome

#import library
from janome.tokenizer import Tokenizer
sample = oppenheim[400:402]
sample.append('not japanese')

def text_prep(arg):
    prep_text=[]

    for line in arg:
        words=""
        for w in line:     
            if is_japanese(w)==True:
                words = words + " " + w
        if words != "":
            prep_text.append([words.strip()])
    return prep_text


text = oppenheim.copy()

In [33]:
#import library
from janome.tokenizer import Tokenizer

def cleanText(arg):
    
    punctuation =  ['。', '、', '？', '！', '「', '」', '『', '』', '（', '）', '；', '：', '-']
    stop_words = ['の', 'に', 'は', 'を', 'た', 'が', 'で', 'て', 'と', 'し', 'れ', 'さ', 'ある', 'いる', 'も', 'する', 'から', 'な', 'こと', 'として', 'い', 'や', 'する', 'など', 'なり', 'なく', 'まで', 'だ', 'へ', 'か', 'だっ', 'その', 'あっ', 'よう', 'また', 'もの', 'という', 'あり', 'まし', 'ませ', 'う', 'ない', 'ながら', 'なけれ', 'なし', 'ず', 'なっ', 'れる', 'られ', 'なる', 'べき', 'ほど', 'ます', 'てる', 'なら', 'せる', 'され', 'して']
    clean_list = []
    prep_text=[]
    
    if type(arg) != list:
        arg = [arg]
        
    for line in arg:
        words=""
        for w in line:     
            if is_japanese(w)==True:
                words = words + " " + w
            
        if words == "":
            prep_text.append([])
        else:    
            prep_text.append([words.strip()])
            
        if prep_text == [[]]:
            return []
    
    for element in prep_text:
        for x in element:
            words = []
            tokenizer = Tokenizer()
            tokens = [token.surface for token in tokenizer.tokenize(x)]
        
        for w in tokens:
            if w != " ":
                if w not in punctuation and w not in stop_words:
                    words.append(w)               
        clean_list.append(words)
        
    return clean_list

In [35]:
# cleaning text and assigning to 'data'
data = cleanText(text)

### Probability Calculation: 

Calculated the probability of occurrence for each word within the text. This was achieved by dividing the frequency of each word by the total number of words present in the text, using a selection of functions provided by the NLTK library:

__from nltk.lm.preprocessing import padded_everygram_pipeline__
    Default preprocessing for a sequence of sentences.

    Creates two iterators:
        - sentences padded and turned into sequences of `nltk.util.everygrams`
        - sentences padded as above and chained together for a flat stream of words  
        :param order: Largest ngram length produced by `everygrams`.
        :param text: Text to iterate over. Expected to be an iterable of sentences.
        :type text: Iterable[Iterable[str]]
        :return: iterator over text as ngrams, iterator over text as vocabulary data

__from nltk.lm import MLE__    
    Class for providing MLE ngram model scores.
    
    _unmasked_score(word, context=None)_
        Returns the MLE score for a word given a context.
        Args:
            - word is expected to be a string
            - context is expected to be something reasonably convertible to a tuple

In [41]:
import nltk
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE

#initiating train_data and padded_vocab with padded_everygram_pipeline
n=1
train_data, padded_vocab = padded_everygram_pipeline(n, data)

padded_vocab = list(padded_vocab)
unigram_model = MLE(n)
unigram_model.fit(train_data, padded_vocab)

In [43]:
def unique_list(arg):
 
    unique_list = []
     
    for x in arg:
        if x not in unique_list:
            unique_list.append(x)
    return unique_list

unique_vocab = unique_list(padded_vocab)

In [45]:
#function to make dictionary from model.score
def score_dict(arg):
    score_dict = {}
    for x in arg:
        score_dict[x] = unigram_model.score(x)
    return score_dict

unigram_dict = score_dict(unique_vocab)

In [51]:
# listing the top ten 
unigram_probs = {}
for key in sorted(unigram_dict, key=unigram_dict.get, reverse=True):
    unigram_probs[key] = unigram_dict[key]

list(unigram_probs.items())[0:10]

[('っ', 0.21357865217420405),
 ('わ', 0.20985737328576878),
 ('男', 0.20729732081305927),
 ('入', 0.2072186786782767),
 ('る', 0.004844020855224849),
 ('ら', 0.004281813253374919),
 ('こ', 0.004206517592412875),
 ('ま', 0.0032845640548553987),
 ('ー', 0.003261138738111652),
 ('あ', 0.003035251755225519)]